In [ ]:
# All package imports (run this cell first)
import sys
import json
from pathlib import Path

import torch
from datasets import load_from_disk
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import PeftModel, get_peft_model, LoraConfig, TaskType

## Kernel check (use root .venv)

Run this cell first to confirm the notebook is using the project's root `.venv`.

In [ ]:
# Kernel verification
_venv_ok = "My-Crew-Manager" in sys.executable and ".venv" in sys.executable
print(f"Python: {sys.executable}")
print(f"Using root .venv: {'✓ Yes' if _venv_ok else '✗ No – select Kernel → Python (My-Crew-Manager .venv)'}")

# Model Part 2 – Backlog Generation

From structured Part 1 (JSON) → backlog text (Epic → Sub-Epic → User Story → Task).

Requires model2_part1_to_backlog.jsonl from build_synthetic. Run Model 1 notebook Step 1 first, or build_synthetic separately.

## Setup paths

In [ ]:
# Resolve AI root
_cwd = Path.cwd()
_ai_root = _cwd if (_cwd / "llms").exists() else (_cwd / "AI" if (_cwd / "AI").exists() else _cwd)
if str(_ai_root) not in sys.path:
    sys.path.insert(0, str(_ai_root))

FINE_TUNE_DIR = _ai_root / "llms" / "fine_tune"
DATASET_DIR = FINE_TUNE_DIR / "dataset"
TOKENIZED_DIR = FINE_TUNE_DIR / "tokenized"
OUTPUT_DIR = FINE_TUNE_DIR / "qwen_model2_backlog_lora"

print(f"AI root: {_ai_root}")
print(f"Dataset: {DATASET_DIR}")
print(f"Output: {OUTPUT_DIR}")

## Step 1: Verify dataset

Ensure model2_part1_to_backlog.jsonl exists (from build_synthetic).

In [ ]:
p = DATASET_DIR / "model2_part1_to_backlog.jsonl"
if not p.exists():
    print("Run build_synthetic first: python -m llms.fine_tune.build_synthetic")
else:
    count = len([ln for ln in p.read_text(encoding="utf-8").split("\n") if ln.strip()])
    print(f"model2_part1_to_backlog.jsonl: {count} examples")

## Step 2: Prepare tokenized dataset

In [ ]:
from llms.fine_tune.prepare_dataset import prepare_model2_dataset

tokenized_path = TOKENIZED_DIR / "tokenized_model2_qwen"
MAX_LENGTH = 768

if tokenized_path.exists():
    dataset = load_from_disk(str(tokenized_path))
    print(f"Loaded tokenized dataset from {tokenized_path}")
else:
    dataset = prepare_model2_dataset(
        model_name="qwen",
        max_length=MAX_LENGTH,
        output_dir=str(tokenized_path),
    )
print(f"Dataset size: {len(dataset)}")

## Step 3: Load model & apply LoRA

In [ ]:
MODEL_ID = "Qwen/Qwen2-0.5B-Instruct"
BATCH_SIZE = 2
EPOCHS = 3

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    trust_remote_code=True,
)
if torch.cuda.is_available():
    model = model.to("cuda")

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## Step 4: Train

In [ ]:
import os
os.environ.setdefault("TENSORBOARD_LOGGING_DIR", str(OUTPUT_DIR / "logs"))

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

trainer.train()

## Step 5: Save adapter

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print(f"Saved adapter and tokenizer to {OUTPUT_DIR}")
print("Set PEFT_ADAPTER_PATH_BACKLOG in AI/.env to use this adapter:")
print("  PEFT_ADAPTER_PATH_BACKLOG=llms/fine_tune/qwen_model2_backlog_lora")

## Step 6: Quick inference test

In [ ]:
from llms.backlog_llm import parse_backlog

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    trust_remote_code=True,
)
model_infer = PeftModel.from_pretrained(base_model, str(OUTPUT_DIR))
if torch.cuda.is_available():
    model_infer = model_infer.to("cuda")

# Sample Part 1 JSON (from Model 1 output)
sample_part1 = json.dumps({
    "summary": "Task management web app for small teams.",
    "roles": ["PM", "Backend Dev", "Frontend Dev", "QA"],
    "features": ["Kanban", "Real-time updates"],
    "goals": [{"epic": "Build API", "role": "Backend Dev"}, {"epic": "Build dashboard", "role": "Frontend Dev"}],
    "timeline": {"week1": ["Setup"], "week2": ["API"], "week3": ["Dashboard"], "week4": ["Test"]}
})

test_prompt = f"Given this structured project overview (JSON), generate an Agile backlog with Epic → Sub-Epic → User Story → Task hierarchy. Output in the exact text format: Epic X: ...\n -Sub-Epic X.1: ...\n  -Task X.1.1.1: ...\n\nInput JSON:\n{sample_part1}\n\nBacklog:"
inputs = tokenizer(test_prompt, return_tensors="pt")
if torch.cuda.is_available():
    inputs = {k: v.cuda() for k, v in inputs.items()}
outputs = model_infer.generate(**inputs, max_new_tokens=512, do_sample=True, temperature=0.4)
response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("Generated backlog:")
print(response[:600])
backlog_model = parse_backlog(response)
print(f"\\nParsed: {len(backlog_model.epics)} epics")